In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O
import os

# ⚠️ 不要对 /kaggle/input 做 os.walk 全量遍历——BirdCLEF 音频有几十万 .ogg,
#    全量遍历会卡几分钟并刷屏。只看顶层 + 一层子目录即可, 瞬间完成。
print("📂 /kaggle/input 结构 (顶层 + 一层):")
for name in sorted(os.listdir("/kaggle/input")):
    sub = os.path.join("/kaggle/input", name)
    print(f"📁 {name}")
    if os.path.isdir(sub):
        for s in sorted(os.listdir(sub))[:12]:
            print(f"   - {s}")

# You can write up to 20GB to /kaggle/working/ that gets preserved as output when
# you create a version using "Save & Run All".

In [ ]:
import os, librosa, numpy as np, pandas as pd

from tqdm.notebook import tqdm

import lightgbm as lgb

print("✅ 依赖库导入成功！")

In [ ]:
import os, re, json, librosa, numpy as np, pandas as pd
from tqdm.notebook import tqdm
import lightgbm as lgb
from sklearn.metrics import accuracy_score

# ================= 0. 配置 =================
# 输出目录与 YAMNet 的 /kaggle/working/yamnet 平级, 产物命名对齐, 便于横向对比。
OUT_DIR = "/kaggle/working/lightgbm"
os.makedirs(OUT_DIR, exist_ok=True)
N_FOLDS = 5          # 使用固定的 5 折划分，结果可与其他模型逐折比较
SEED = 42            # 控制 LightGBM 的随机采样，使重复运行尽量可复现
SR = 22050           # 所有音频统一重采样到 22.05 kHz，避免不同采样率造成特征维度/尺度不一致
MAX_DURATION = 30    # 每段最多读取前 30 秒，限制单文件的计算时间和内存占用

# ================= 1. 扫描音频目录 + CSV（一次扫描, 5 折复用） =================
# BirdCLEF 实际结构: .ogg 嵌套在 species 子目录里, train_audio 顶层只有子目录;
# 2021 用 train_short_audio, 2022~2026 用 train_audio。遍历遇音频目录剪枝, 不进几十万 .ogg。
# Kaggle 把竞赛/数据集嵌套挂在 /kaggle/input/competitions 和 /kaggle/input/datasets 下,
# os.walk 会自动穿透这层嵌套找到 train_audio。年份从路径里抓 20XX, 兼容
# birdclef-2024 / birdclefplus-2025 / birdclefplus-2026 等命名。
INPUT_BASE = "/kaggle/input"
year2root = {}
csv_paths = {}

if not os.path.exists(INPUT_BASE):
    raise SystemExit(f"❌ {INPUT_BASE} 不存在, 数据集没有挂载。")

target_csvs = {"ml_test.csv"}
for f in range(1, N_FOLDS + 1):
    target_csvs.add(f"ml_cv_fold{f}_train.csv")
    target_csvs.add(f"ml_cv_fold{f}_val.csv")

for root, dirs, files in os.walk(INPUT_BASE, topdown=True):
    for name in ("train_audio", "train_short_audio"):
        if name not in dirs:
            continue
        cand = os.path.join(root, name)
        try:
            items = os.listdir(cand)
        except Exception:
            items = []
        has_ogg = any(x.lower().endswith(".ogg") for x in items)
        has_subdir = any(os.path.isdir(os.path.join(cand, x)) for x in items)
        if not items or not (has_ogg or has_subdir):
            continue
        m = re.search(r"(20\d{2})", root)
        key = int(m.group(1)) if m else root
        if key not in year2root:
            year2root[key] = cand
    dirs[:] = [d for d in dirs if d not in ("train_audio", "train_short_audio")]
    for f in files:
        if f in target_csvs and f not in csv_paths:
            csv_paths[f] = os.path.join(root, f)

if not year2root:
    # 扫不到 train_audio 时, 打印最多 4 层目录树, 方便定位数据到底挂哪了
    print("❌ 未找到 train_audio 目录。当前 /kaggle/input 结构 (最多 4 层):")
    for root, dirs, files in os.walk(INPUT_BASE, topdown=True):
        depth = root[len(INPUT_BASE):].count(os.sep)
        if depth >= 4:
            dirs[:] = []   # 不再下钻, 避免陷进几十万 .ogg
            continue
        print(f"  {root}/  dirs={sorted(dirs)[:15]}  files={sorted(files)[:5]}")
    raise SystemExit("❌ 未找到 train_audio 目录 (见上方结构)。请确认右侧 Input 已添加 BirdCLEF 竞赛数据。")
print(f"✅ 音频目录 {len(year2root)} 个:")
for y, r in sorted(year2root.items(), key=lambda kv: str(kv[0])):
    print(f"   - {y}: {r}")
all_roots = list(year2root.values())

missing_csv = [c for c in sorted(target_csvs) if c not in csv_paths]
if missing_csv:
    raise SystemExit(f"❌ 未找到 CSV: {missing_csv}\n   input 下一层: {os.listdir(INPUT_BASE)}")
print(f"✅ 找到全部 {len(target_csvs)} 个 CSV (位于 {os.path.dirname(next(iter(csv_paths.values())))})。")

# ================= 2. 共享标签映射（与 YAMNet 完全一致） =================
# classes = sorted(fold1 train + fold1 val + test 的 primary_label 并集)
# 顺序与 YAMNet 相同 -> 两模型的 classes 数组对齐, 整数下标才可直接横比。
label_series = pd.concat([
    pd.read_csv(csv_paths["ml_cv_fold1_train.csv"], usecols=["primary_label"])["primary_label"],
    pd.read_csv(csv_paths["ml_cv_fold1_val.csv"], usecols=["primary_label"])["primary_label"],
    pd.read_csv(csv_paths["ml_test.csv"], usecols=["primary_label"])["primary_label"],
]).astype(str)
classes = sorted(label_series.unique().tolist())
label2idx = {c: i for i, c in enumerate(classes)}
with open(os.path.join(OUT_DIR, "label_map.json"), "w", encoding="utf-8") as fp:
    json.dump({"label2idx": label2idx,
               "idx2label": {str(i): c for c, i in label2idx.items()}},
              fp, ensure_ascii=False, indent=2)
print(f"✅ 标签映射: {len(classes)} 类, 存 {OUT_DIR}/label_map.json")

# ================= 3. 特征提取函数 + 路径解析 =================
FEATURE_NAMES = (["duration", "zcr", "spectral_centroid", "spectral_bandwidth",
                  "spectral_rolloff", "rms"]
                 + [f"mfcc_mean_{i}" for i in range(13)]
                 + [f"mfcc_std_{i}" for i in range(13)])

def parse_years(s):
    """把 CSV 中的 source_year 字段解析为年份列表。

    数据中可能使用逗号或分号分隔多个来源年份；无法解析的内容会被忽略。
    年份按从新到旧排序，使路径搜索优先尝试较新的竞赛数据目录。
    """
    if pd.isna(s) or not str(s).strip():
        return []
    years = []
    for part in str(s).replace(";", ",").split(","):
        try:
            years.append(int(part.strip()))
        except ValueError:
            pass  # 容忍空字符串或非数字标记，不因单条脏数据中断整个任务
    return sorted(years, reverse=True)

def find_audio(row):
    """根据一行 CSV 元数据定位对应的音频文件。

    不同年份的 BirdCLEF 数据目录结构并不完全一致，因此依次尝试：
    1. root/<CSV 中的 filename>；
    2. root/<primary_label>/<文件 basename>；
    3. root/<文件 basename>。
    优先搜索 source_year 指定的目录，若未命中再遍历其他年份目录。
    找到第一个真实存在的文件时返回完整路径，否则返回 None。
    """
    # CSV 可能来自 Windows，先把反斜杠统一为正斜杠，便于在 Kaggle Linux 环境解析。
    fname = str(row["filename"]).replace("\\", "/")
    base = fname.split("/")[-1]
    pl = str(row.get("primary_label", "")).strip()
    roots_to_try = []
    for y in parse_years(row.get("source_year", None)):
        if y in year2root and year2root[y] not in roots_to_try:
            roots_to_try.append(year2root[y])
    for r in all_roots:
        if r not in roots_to_try:
            roots_to_try.append(r)
    candidates = []
    for root in roots_to_try:
        for cand in (os.path.join(root, fname),
                     os.path.join(root, pl, base) if pl else None,
                     os.path.join(root, base)):
            if cand and cand not in candidates:
                candidates.append(cand)
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

def _features_from_wave(y):
    """将一维音频波形转换为固定长度的 32 维特征向量。

    特征组成：1 个时长 + 5 个全局声学统计量 + 13 个 MFCC 均值
    + 13 个 MFCC 标准差。无论原始音频多长，输出形状始终为 (32,)，
    因而可以直接作为 LightGBM 的表格型输入。干净音频和加噪音频都调用
    本函数，确保鲁棒性实验只改变波形，而不改变特征提取流程。
    """
    # MFCC 描述短时频谱包络，常用于区分不同声源或鸟鸣音色。
    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=13)
    feats = [
        len(y) / SR,  # 实际加载到的音频时长（最长为 MAX_DURATION）
        np.mean(librosa.feature.zero_crossing_rate(y=y)),      # 过零率：粗略反映高频/噪声程度
        np.mean(librosa.feature.spectral_centroid(y=y, sr=SR)), # 频谱质心：声音的“明亮度”
        np.mean(librosa.feature.spectral_bandwidth(y=y, sr=SR)),# 频谱带宽：能量在频域的分散程度
        np.mean(librosa.feature.spectral_rolloff(y=y, sr=SR)),  # 滚降频率：累计频谱能量的高频边界
        # 新版 librosa 将 y 设为仅限关键字参数，因此必须写成 rms(y=y)。
        np.mean(librosa.feature.rms(y=y)),                       # 均方根能量：整体响度/强度
    ]
    # 对每个 MFCC 系数沿时间轴求均值和标准差，压缩为固定长度的统计特征。
    for i in range(13):
        feats.append(float(mfcc[i].mean()))
    for i in range(13):
        feats.append(float(mfcc[i].std()))
    return np.array(feats, dtype=np.float32)

def extract_features(audio_path):
    """读取单个音频并提取特征；文件损坏或过短时返回 None。

    sr=SR 会统一重采样，mono=True 会把多声道合并为单声道，duration 则限制
    最多读取 MAX_DURATION 秒。这里返回 None 而不是抛出异常，是为了让批量处理
    能跳过少量异常文件并继续执行。
    """
    try:
        y, _ = librosa.load(audio_path, sr=SR, mono=True, duration=MAX_DURATION)
        if len(y) < SR * 0.1:  # 少于 0.1 秒时，统计特征通常不稳定，直接视为无效样本
            return None
    except Exception:
        return None
    return _features_from_wave(y)

# ================= 4. 按 filename 缓存特征（5 折 + 测试共享, 只算一次） =================
# fold1 train+val = 4780 全集, + test 1196 = 5976 唯一; 5 折只是对 4780 重切, 缓存全覆盖。
CACHE_PATH = os.path.join(OUT_DIR, "features_cache.npz")
# 两个字典都以 filename 为键：一个保存 32 维特征，一个保存整数类别标签。
# 这样后续任意折只需按文件名查表，无需重复读取和分析音频。
fn2feat, fn2y = {}, {}
needed_rows = {}   # filename -> (实际音频路径, 字符串类别标签)

def collect(df):
    """收集 DataFrame 中尚未登记的唯一音频，并提前解析其真实路径。"""
    for _, row in df.iterrows():
        fn = str(row["filename"])
        if fn not in needed_rows:
            needed_rows[fn] = (find_audio(row), str(row["primary_label"]))

# fold1 的 train+val 已覆盖完整训练集；再加入 test 即可覆盖后续所有计算样本。
for csv_name in ["ml_cv_fold1_train.csv", "ml_cv_fold1_val.csv", "ml_test.csv"]:
    collect(pd.read_csv(csv_paths[csv_name]))
print(f"✅ 去重后需提取特征 {len(needed_rows)} 条")

if os.path.exists(CACHE_PATH):
    # allow_pickle=True 用于兼容字符串文件名数组；缓存只读取本任务自己生成的文件。
    data = np.load(CACHE_PATH, allow_pickle=True)
    for i, fn in enumerate(data["filenames"]):
        sfn = str(fn)
        fn2feat[sfn] = data["X"][i]
        fn2y[sfn] = int(data["y"][i])
    print(f"✅ 缓存命中 {len(fn2feat)} 条")

need = [fn for fn in needed_rows if fn not in fn2feat]
if need:
    print(f"🚀 补算 {len(need)} 条特征 ...")
    for i, fn in enumerate(tqdm(need)):
        path, lab = needed_rows[fn]
        if path is None:
            continue
        feat = extract_features(path)
        if feat is None:
            continue
        fn2feat[fn] = feat
        fn2y[fn] = label2idx[lab]
    all_fn = list(fn2feat.keys())
    np.savez(CACHE_PATH,
             filenames=np.array(all_fn),
             X=np.stack([fn2feat[f] for f in all_fn]).astype(np.float32),
             y=np.array([fn2y[f] for f in all_fn], dtype=np.int64),
             feature_names=np.array(FEATURE_NAMES))
    print(f"✅ 缓存已写: {CACHE_PATH}")

def split_xy(df):
    """按 DataFrame 原始行顺序组装模型输入、标签和文件名。

    返回值：
      X: shape=(有效样本数, 32) 的 float32 特征矩阵；
      y: shape=(有效样本数,) 的 int64 类别下标；
      filenames: 与 X、y 严格对齐的文件名列表。
    未成功提取特征的文件不会出现在缓存中，因此会在这里被跳过。
    """
    Xs, ys, fns = [], [], []
    for _, r in df.iterrows():
        fn = str(r["filename"])
        if fn in fn2feat:
            Xs.append(fn2feat[fn])
            ys.append(label2idx[str(r["primary_label"])])
            fns.append(fn)
    return (np.array(Xs, dtype=np.float32),
            np.array(ys, dtype=np.int64),
            fns)

# ================= 5. 5 折训练 + 测试评估 =================
# 每一折使用不同的训练/验证切分，但都在同一独立测试集上计算准确率。
# clean_accs 最终保存 5 个测试准确率，用于计算跨折均值和标准差。
clean_accs = []
for fold in range(1, N_FOLDS + 1):
    print(f"\n########## FOLD {fold}/{N_FOLDS} ##########")
    df_train = pd.read_csv(csv_paths[f"ml_cv_fold{fold}_train.csv"])
    df_val = pd.read_csv(csv_paths[f"ml_cv_fold{fold}_val.csv"])
    df_test = pd.read_csv(csv_paths["ml_test.csv"])
    X_tr, y_tr, _ = split_xy(df_train)
    X_va, y_va, _ = split_xy(df_val)
    X_te, y_te, te_fns = split_xy(df_test)
    print(f"   train={len(X_tr)} val={len(X_va)} test={len(X_te)}")

    fold_out = os.path.join(OUT_DIR, f"fold{fold}")
    os.makedirs(fold_out, exist_ok=True)

    # 用底层 lgb.train 而非 LGBMClassifier: sklearn 包装内部用 LabelEncoder 只认识
    # 训练集出现过的类, 验证集里有"训练集没见过的类"(鸟种级折切, 稀有种只出现在某折 val)
    # 会报 "y contains previously unseen labels"。底层 API 显式给 num_class=全类数,
    # 直接吃 0..N-1 整数标签, 稀有种没训练样本也能正常 argmax 预测——与 YAMNet 的
    # 固定 N 个 softmax 输出口径一致。存出的 .txt 正是 cell3/cell4 用 Booster 加载的格式。
    train_set = lgb.Dataset(X_tr, label=y_tr)
    val_set = lgb.Dataset(X_va, label=y_va, reference=train_set)
    params = {
        "objective": "multiclass",       # 多分类概率输出，每行得到 num_class 个概率
        "num_class": len(classes),        # 固定为全局类别总数，包含本折训练集未出现的类别
        "learning_rate": 0.05,            # 较小步长通常更稳健，但需要更多迭代轮数
        "num_leaves": 31,                 # 单棵树最大叶子数，控制模型表达能力
        "feature_fraction": 0.8,          # 每棵树随机使用 80% 特征，降低过拟合风险
        "bagging_fraction": 0.8,          # 每次装袋随机使用 80% 训练样本
        "bagging_freq": 5,                # 每 5 轮重新进行一次样本抽样
        "metric": "multi_logloss",       # 早停依据：验证集多分类交叉熵
        "seed": SEED + fold,              # 各折种子可复现且彼此不同
        "verbose": -1,                    # 关闭 LightGBM 冗余日志，由下方回调控制输出
    }
    booster = lgb.train(
        params,
        train_set,
        num_boost_round=1000,              # 最大轮数；通常会被 early_stopping 提前终止
        valid_sets=[train_set, val_set],
        valid_names=["train", "val"],
        callbacks=[
            # 验证损失连续 50 轮没有改善时停止，并保留最佳轮次。
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=0),   # 不逐轮打印，保持 Kaggle 输出简洁
        ],
    )
    booster.save_model(os.path.join(fold_out, "lightgbm_model.txt"))

    pred = np.argmax(booster.predict(X_te), axis=1)
    acc = accuracy_score(y_te, pred)
    clean_accs.append(float(acc))
    print(f"   fold{fold} 测试准确率 = {acc:.4f}")

    np.savez(os.path.join(fold_out, "test_predictions.npz"),
             y_true=y_te, y_pred=pred,
             classes=np.array(classes),
             test_filenames=np.array(te_fns))

# ================= 6. 汇总（与 YAMNet 同格式） =================
cv_per_fold = pd.DataFrame({"fold": range(1, N_FOLDS + 1), "clean_acc": clean_accs})
cv_per_fold.to_csv(os.path.join(OUT_DIR, "cv_per_fold.csv"), index=False)
clean_arr = np.array(clean_accs)
summary = pd.DataFrame([{"metric": "clean_acc",
                         "mean": float(clean_arr.mean()),
                         "std": float(clean_arr.std(ddof=0))}])
summary.to_csv(os.path.join(OUT_DIR, "cv_summary.csv"), index=False)
print(f"\n🏆 LightGBM 5 折 clean accuracy: {clean_arr.mean():.4f} ± {clean_arr.std(ddof=0):.4f}")
print(f"   逐折: {[f'{a:.4f}' for a in clean_accs]}")
print(f"   产物目录: {OUT_DIR}")
print("\n下一步: 运行下一格 (cell3) 做噪声鲁棒性评估。")

In [ ]:
# ================= 噪声鲁棒性评估（与 YAMNet cell3 对称, 可直接横比） =================
# 本格在 cell7 之后运行, 复用其命名空间 (OUT_DIR/CACHE_PATH/csv_paths/year2root/find_audio/
# _features_from_wave/label2idx/classes/N_FOLDS/SEED/SR/MAX_DURATION)。
#
# 叠噪规范与 YAMNet 完全一致 (HANDOFF §8):
#   高斯白噪声; SNR_dB = 10*log10(P_signal/P_noise); 叠后峰值归一化到 [-1,1]; 固定种子 42。
#   4 档: clean / 5dB / 0dB / -5dB。
# LightGBM 的差异: 叠噪后重新提取 32 维手工特征 (与训练同流程), 而非 YAMNet 嵌入。
# clean 档复用训练缓存特征, 保证基线与 cell7 的 clean 准确率一致。
#
# 产物 (与 YAMNet 同格式):
#   fold{N}/noise_results.npz, cv_noise_per_fold.csv, 往 cv_summary.csv 追加 acc_5dB/0dB/-5dB 行。

import os, numpy as np, pandas as pd, librosa
from tqdm.notebook import tqdm
from lightgbm import Booster

SNR_TIERS = ["clean", "5dB", "0dB", "-5dB"]
SNR_VALS = [("5dB", 5.0), ("0dB", 0.0), ("-5dB", -5.0)]

def add_gaussian_noise(waveform, snr_db, rng):
    """按指定信噪比向波形叠加高斯白噪声，并执行峰值归一化。

    SNR(dB)=10*log10(P_signal/P_noise)，因此已知信号平均功率后可以反推出
    目标噪声功率。snr_db 越小，噪声越强；-5 dB 表示噪声功率已经高于信号。
    rng 由外部传入，保证所有实验使用固定且可复现的随机序列。
    """
    # 加极小值避免全静音波形造成除零。
    signal_power = np.mean(waveform ** 2) + 1e-12
    noise_power = signal_power / (10 ** (snr_db / 10))
    # 高斯分布的标准差等于功率（方差）的平方根。
    noise = rng.normal(0, np.sqrt(noise_power), size=waveform.shape).astype(np.float32)
    noisy = waveform + noise
    # 统一缩放到 [-1, 1]，防止叠噪后幅值溢出，同时保持波形相对结构。
    peak = np.max(np.abs(noisy)) + 1e-9
    return (noisy / peak).astype(np.float32)

# ---- 1. 取测试集, 复用 cell7 的 find_audio 解析路径 (只对训练时已命中缓存的条做噪声) ----
df_test = pd.read_csv(csv_paths["ml_test.csv"])
test_rows = []
for _, row in df_test.iterrows():
    fn = str(row["filename"])
    if fn in fn2feat:   # 保证 clean 基线与训练一致
        test_rows.append((fn, find_audio(row), str(row["primary_label"])))
print(f"✅ 噪声测试集 {len(test_rows)} 条 (与 clean 基线同一批样本)")

# ---- 2. 各档特征: clean 复用缓存; 5/0/-5dB 叠噪后重算手工特征 ----
rng = np.random.default_rng(SEED)   # 固定种子, 5 折噪声一致且与 YAMNet 同种子
feats_by_snr = {k: [] for k in SNR_TIERS}
y_true = []
for fn, path, lab in tqdm(test_rows, desc="噪声特征"):
    y_true.append(label2idx[lab])
    feats_by_snr["clean"].append(fn2feat[fn])          # 复用训练缓存
    try:
        y, _ = librosa.load(path, sr=SR, mono=True, duration=MAX_DURATION)
    except Exception:
        y = None
    if y is None or len(y) < SR * 0.1:
        # 取不到波形的条, 噪声档用 clean 特征兜底, 保持 shape 对齐
        for k, _ in SNR_VALS:
            feats_by_snr[k].append(fn2feat[fn])
        continue
    for k, sv in SNR_VALS:
        feats_by_snr[k].append(_features_from_wave(add_gaussian_noise(y, sv, rng)))

X_by_snr = {k: np.array(v, dtype=np.float32) for k, v in feats_by_snr.items()}
y_true = np.array(y_true, dtype=np.int64)
test_filenames = np.array([fn for fn, _, _ in test_rows])

# ---- 3. 各折模型在 4 档上预测 (加载 cell7 存的 lightgbm_model.txt) ----
rows = []
for fold in range(1, N_FOLDS + 1):
    model_path = os.path.join(OUT_DIR, f"fold{fold}", "lightgbm_model.txt")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"fold{fold} 模型不存在: {model_path}; 请先跑 cell7。")
    booster = Booster(model_file=model_path)
    acc_by_snr, preds_by_snr = {}, {}
    for k in SNR_TIERS:
        pred = np.argmax(booster.predict(X_by_snr[k]), axis=1)
        preds_by_snr[k] = pred
        acc_by_snr[k] = float(np.mean(pred == y_true))
    print(f"[噪声] fold{fold}: " + "  ".join(f"{k}={v:.4f}" for k, v in acc_by_snr.items()))

    np.savez(os.path.join(OUT_DIR, f"fold{fold}", "noise_results.npz"),
             snr_tiers=np.array(SNR_TIERS),
             acc=np.array([acc_by_snr[s] for s in SNR_TIERS]),
             y_true=y_true,
             preds_clean=preds_by_snr["clean"],
             preds_5dB=preds_by_snr["5dB"],
             preds_0dB=preds_by_snr["0dB"],
             preds_n5dB=preds_by_snr["-5dB"])
    rows.append({"fold": fold, **{f"acc_{k}": acc_by_snr[k] for k in SNR_TIERS}})

# ---- 4. 汇总 (与 YAMNet 同格式: cv_noise_per_fold.csv + 追加 cv_summary.csv) ----
per_fold = pd.DataFrame(rows)
per_fold.to_csv(os.path.join(OUT_DIR, "cv_noise_per_fold.csv"), index=False)

noise_metrics = [f"acc_{k}" for k in SNR_TIERS]
summary_path = os.path.join(OUT_DIR, "cv_summary.csv")
existing = pd.read_csv(summary_path) if os.path.exists(summary_path) \
    else pd.DataFrame(columns=["metric", "mean", "std"])
existing = existing[~existing["metric"].isin(noise_metrics)]   # 去旧噪声行, 支持重跑
new_rows = pd.DataFrame([{
    "metric": m,
    "mean": float(per_fold[m].mean()),
    "std": float(per_fold[m].std(ddof=0)),
} for m in noise_metrics])
pd.concat([existing, new_rows], ignore_index=True).to_csv(summary_path, index=False)

print(f"\n🏆 LightGBM 噪声鲁棒性 (mean±std):")
for m in noise_metrics:
    print(f"  {m}: {per_fold[m].mean():.4f} ± {per_fold[m].std(ddof=0):.4f}")
print(f"   汇总已存: {summary_path}")
print("   注: clean 档应与 cell7 的 clean 准确率一致 (同一缓存特征 + 同一模型)。")

In [ ]:
# ================= 推理速度测量（与 YAMNet cell4 对称, 5 折 mean±std） =================
# 本格在 cell7/cell8 之后运行, 复用其命名空间 (OUT_DIR/csv_paths/find_audio/
# _features_from_wave/fn2feat/classes/N_FOLDS/SEED/SR/MAX_DURATION)。
#
# 公平性 (与 YAMNet measure_inference.py 同口径):
#   - 预加载波形 (排除磁盘 I/O), 计时只含"特征提取 + 预测"计算。
#     YAMNet 同样排除 I/O (波形预加载, 嵌入预计算), 故本格对齐, 不含读盘。
#   - 特征提取与折无关 (同一波形->同一特征), 测一次 (对应 YAMNet 的编码器, 共享);
#     booster 预测各折权重不同, 逐折测 (对应 YAMNet 的分类头)。
#   - e2e = 特征提取 + 预测; 5 折 mean±std。
# LightGBM 走 CPU, 不占 GPU 显存, 记为 None/N.A.。

import os, csv, time, numpy as np, pandas as pd, librosa
from tqdm.notebook import tqdm
from lightgbm import Booster

N_SAMPLES = 50
N_WARMUP = 5
MODEL_NAME = "LightGBM"

# ---- 1. 取前 N 条测试样本, 解析路径, 预加载波形 (不计时) ----
df_test = pd.read_csv(csv_paths["ml_test.csv"])
rows = []
for _, r in df_test.iterrows():
    fn = str(r["filename"])
    if fn in fn2feat:          # 只测训练时命中的条 (与 cell8 同口径)
        rows.append((fn, find_audio(r)))
    if len(rows) >= N_SAMPLES:
        break
n_test = len(rows)
print(f"[推理测量] 测试集取前 {n_test} 条, {N_FOLDS} 折, 预热 {N_WARMUP}")

print(f"[推理测量] 预加载 {n_test} 条波形 (排除 I/O) ...")
waveforms = []
for fn, path in tqdm(rows, desc="预加载波形"):
    try:
        y, _ = librosa.load(path, sr=SR, mono=True, duration=MAX_DURATION)
    except Exception:
        y = np.zeros(int(SR * MAX_DURATION), dtype=np.float32)
    waveforms.append(y)

# ---- 2. 测量特征提取延迟 (与折无关, 测一次; 对应 YAMNet 编码器) ----
# 首次调用 librosa.feature 会触发 numba JIT 编译, 预热覆盖该开销。
print(f"\n[推理测量] 测量特征提取 ({N_WARMUP} 预热 + {n_test} 正式) ...")
for _ in range(N_WARMUP):
    _ = _features_from_wave(waveforms[0])
feat_times = []
for i in range(n_test):
    t0 = time.perf_counter()
    _ = _features_from_wave(waveforms[i])
    feat_times.append((time.perf_counter() - t0) * 1000)
feat_arr = np.array(feat_times)
print(f"  特征提取: {feat_arr.mean():.1f} ± {feat_arr.std():.1f} ms/条")

# ---- 3. 预计算 50 条特征 (各折共享, 避免重复提取) ----
cached_feats = np.stack([_features_from_wave(w) for w in waveforms]).astype(np.float32)

# ---- 4. 逐折测量 booster 预测延迟 (对应 YAMNet 分类头) ----
fold_head_means, fold_head_stds, fold_e2e_means = [], [], []
for fold in range(1, N_FOLDS + 1):
    model_path = os.path.join(OUT_DIR, f"fold{fold}", "lightgbm_model.txt")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"fold{fold} 模型不存在: {model_path}; 请先跑 cell7。")
    booster = Booster(model_file=model_path)
    print(f"\n[推理测量] fold{fold} 预测测量 ({N_WARMUP} 预热 + {n_test} 正式) ...")
    for _ in range(N_WARMUP):
        _ = np.argmax(booster.predict(cached_feats[0:1]), axis=1)
    head_times = []
    for i in range(n_test):
        t0 = time.perf_counter()
        _ = np.argmax(booster.predict(cached_feats[i:i + 1]), axis=1)
        head_times.append((time.perf_counter() - t0) * 1000)
    head_arr = np.array(head_times)
    fold_head_means.append(float(head_arr.mean()))
    fold_head_stds.append(float(head_arr.std()))
    fold_e2e_means.append(float(feat_arr.mean() + head_arr.mean()))
    print(f"  fold{fold} 预测: {head_arr.mean():.1f} ± {head_arr.std():.1f} ms/条")
    print(f"  fold{fold} 端到端: {feat_arr.mean() + head_arr.mean():.1f} ms/条")

head_means_arr = np.array(fold_head_means)
e2e_means_arr = np.array(fold_e2e_means)

print(f"\n{'='*60}")
print(f"[推理速度] 5 折汇总 (mean ± std):")
print(f"  端到端 (波形→特征→预测): {e2e_means_arr.mean():.1f} ± {e2e_means_arr.std():.1f} ms/条")
print(f"    其中 特征提取:          {feat_arr.mean():.1f} ± {feat_arr.std():.1f} ms/条 (共享)")
print(f"    其中 booster 预测:      {head_means_arr.mean():.1f} ± {head_means_arr.std():.1f} ms/条 (5 折)")
print(f"  GPU 显存: N.A. (LightGBM 走 CPU 推理)")
print(f"{'='*60}")

# ---- 5. 保存 (与 YAMNet inference_metrics.csv 同构, 便于合表对比) ----
metrics = {
    "model": MODEL_NAME,
    "inference_e2e_mean_ms": round(float(e2e_means_arr.mean()), 1),
    "inference_e2e_std_ms": round(float(e2e_means_arr.std()), 1),
    "feature_mean_ms": round(float(feat_arr.mean()), 1),
    "feature_std_ms": round(float(feat_arr.std()), 1),
    "head_mean_ms": round(float(head_means_arr.mean()), 1),
    "head_std_ms": round(float(head_means_arr.std()), 1),
    "gpu_memory_current_mb": None,
    "gpu_memory_peak_mb": None,
    "n_samples": n_test,
    "n_warmup": N_WARMUP,
    "n_folds": N_FOLDS,
}
out_csv = os.path.join(OUT_DIR, "inference_metrics.csv")
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(metrics.keys()))
    w.writeheader()
    w.writerow(metrics)
print(f"\n[推理测量] 汇总已存: {out_csv}")

detail_csv = os.path.join(OUT_DIR, "inference_details.csv")
with open(detail_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["fold", "e2e_mean_ms", "head_mean_ms", "head_std_ms",
                "feature_mean_ms", "feature_std_ms", "n_samples"])
    for fold in range(1, N_FOLDS + 1):
        w.writerow([fold,
                    round(fold_e2e_means[fold - 1], 1),
                    round(fold_head_means[fold - 1], 1),
                    round(fold_head_stds[fold - 1], 1),
                    round(float(feat_arr.mean()), 1),
                    round(float(feat_arr.std()), 1),
                    n_test])
print(f"[推理测量] 逐折细节已存: {detail_csv}")